# Test de la API desplegada

Vamos a consumir nuestro servicio de **resumen** (summarization) desplegado vía API.

La API (FastAPI) expone:
- `POST /summarize` -> devuelve el resumen en el body de la respuesta
- `POST /summarize/stream` -> transmite el resumen token a token (SSE)
- `GET /docs` -> playground interactivo (Swagger UI)

> Nota: el servicio en fly.io se apaga cuando no se usa (`min_machines_running = 0`),
> así que la **primera** llamada puede tardar unos segundos en "despertar" la máquina.

In [ ]:
# Este cliente solo hace llamadas HTTP, así que únicamente necesita `requests`
# (ya viene incluido en Colab y en Python). No hace falta instalar ningún SDK del servidor.
import requests

## Definamos las variables

La URL del servicio y el texto que queremos resumir.

In [ ]:
# Servicio desplegado en fly.io
BASE_URL = "https://serve-chain-api.fly.dev"

# Para probar contra un servidor local, comenta la línea de arriba y descomenta esta:
# BASE_URL = "http://localhost:8000"

payload = {
    "text": """
El objetivo general del curso de inteligencia generativa es proporcionar
a los participantes un conocimiento exhaustivo y práctico de las tecnologías
y herramientas avanzadas en el campo de la IA generativa.
A través de una combinación de teoría y práctica, el curso busca capacitar a
los estudiantes en la comprensión y aplicación de diferentes modelos de IA,
como la generación de texto y de imágenes, el procesamiento de voz,
y el desarrollo de agentes inteligentes. Se enfatiza en el uso de
herramientas específicas como Github Copilot, LangChain,
y modelos como GPT-4 y DALL-E. Además, el curso abarca aspectos cruciales como la calidad,
seguridad, ética y legislación en el desarrollo de IA,
preparando a los participantes para enfrentar desafíos reales en el
ámbito de la inteligencia generativa aplicada al desarrollo de software.
"""
}

## Método 1: llamar al endpoint `/summarize`

La forma más simple: una petición `POST` con el texto en el body JSON.

In [ ]:
response = requests.post(f"{BASE_URL}/summarize", json=payload)
response.raise_for_status()
response.json()

## Método 2: respuesta en streaming (`/summarize/stream`)

El endpoint de streaming devuelve el resumen token a token usando Server-Sent Events (SSE).
Útil para mostrar la respuesta en vivo en una interfaz.

In [ ]:
with requests.post(f"{BASE_URL}/summarize/stream", json=payload, stream=True) as r:
    r.raise_for_status()
    for line in r.iter_lines(decode_unicode=True):
        if line and line.startswith("data: "):
            token = line[len("data: "):]
            if token == "[DONE]":
                break
            print(token, end="", flush=True)